In [ ]:
from abc import ABC, abstractmethod 
from math import floor, sqrt, exp

import random

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Para reproducibilidad
semilla = 123
random.seed(semilla)
np.random.seed(semilla)

### Diccionarios en Python

Los diccionarios de Python se llevan a cabo a través de tablas hash. Por ej. podemos almacenar 2 conjuntos usando esta estructura de datos.

In [ ]:
o = {}
o['O1']=(0, 2, 0, 4)
o['O2']=(1, 2, 0, 2)
    
print('O1 = {0}, O2 = {1}'.format(o['O1'], o['O2']))

In [ ]:
o

También es posible hacer lo contrario.

In [ ]:
o2 = {}
o2[(0,2,0,4)]='o1'
o2[(1,2,0,2)]='o2'

print('(0,2,0,4) = {0}'.format(o2[(0,2,0,4)]))
print('(1,2,0,2) = {0}'.format(o2[(1,2,0,2)]))

In [ ]:
o2

Más generalmente podemos crear diccionarios usando cualquier tipo o estructura de datos al que se le pueda aplicar la función hash.

In [ ]:
print('Hash de 1 es {0}'.format(hash(1)))
print('Hash de 3.14 es {0}'.format(hash(3.14)))
print('Hash de \'10\' es {0}'.format(hash('10')))
print('Hash de (1,2) es {0}'.format(hash((1,2))))
print('Hash de (1,2,0,2) es {0}'.format(hash((1,2,0,2))))

### Creando tablas hash en Python

Definimos una clase abstracta para nuestra estructura de datos, especificando sus métodos principales.

In [ ]:
class TablaHash(ABC):
    def __init__(self, n_cubetas):
        self.n_cubetas = n_cubetas
        self.tabla = [[] for i in range(n_cubetas)]

    def __getitem__(self, x):
        return self.tabla[self.h(x)]

    def __repr__(self):
        contenido = ['%d::%s' % (i, self.tabla[i]) for i in range(self.n_cubetas)]
        return "<TablaHash :%s >" % ('\n'.join(contenido))

    def __str__(self):
        contenido = ['%d::%s' % (i, self.tabla[i]) for i in range(self.n_cubetas) if self.tabla[i]]
        return '\n'.join(contenido)

    @abstractmethod
    def h(self, x):
        pass

    @abstractmethod
    def insertar(self, x):
        pass

    @abstractmethod
    def buscar(self, x):
        pass

    @abstractmethod
    def eliminar(self, x):
        pass

Familia H por método de división

Definimos una clase hija de TablaHash que implemente la función del método de división. Para resolver colisiones usamos encadenamiento, esto es, cada cubeta o registro de la tabla es una lista que contiene todos los objetos que son mapeados al índice correspondiente.

In [ ]:
class THMod(TablaHash):
    def h(self, x):
        return x % self.n_cubetas

    def insertar(self, x):
        self.tabla[self.h(x)].append(x)

    def buscar(self, x):
        return x in self.tabla[h(x)]

    def eliminar(self, x):
        self.tabla[self.h(x)].remove(x)

Probemos nuestra tabla con un conjunto 5 de números

In [ ]:
enteros = [1, 10, 36, 78, 93]
n_cubetas = 11

thmod = THMod(n_cubetas)
for i,e in enumerate(enteros):
    thmod.insertar(e)
print(thmod)

¿Qué pasa si tenemos una lista de objetos más grande?

In [ ]:
enteros = [12454, 73523, 9865, 12310, 12, 864189, 882113, 27, 36, 39654, 4, 481, 1155, 634, 69, 782, 1232, 9433]

thmod_mas = THMod(n_cubetas)
for i,e in enumerate(enteros):
    thmod_mas.insertar(e)
print(thmod_mas)

Para evitar colisiones podemos aumentar el tamaño de la tabla. Suponiendo que la función anterior es uniforme, ¿cuál sería la probabilidad de colisión si deseamos almacenar n objetos en una tabla de tamaño m? Podemos calcular esta probabilidad como el complemento de la probabilidad que no haya colisión.

In [ ]:
def cumple(m, n):
    prod = 1
    for i in range(n):
        prod *= (m - i) / m
    return 1.0 - prod

Este problema está relacionado a la paradoja del cumpleaños, en el cual se tienen un conjunto de personas y se busca calular la probabilidad de que al menos 2 personas cumplan años el mismo día. En este caso las probabilidades para diferentes valores de quedan de la siguiente manera:

In [ ]:
pcs = []
n_personas = range(1,90,5)
for n in n_personas:
    pcs.append(cumple(365, n))

plt.plot(range(1,90,5), pcs)
plt.show()

Regresando al problema de las colisiones en nuestra tabla m=11, las probabilidades para serían

In [ ]:
pcs = []
n_personas = range(1,20,5)
for n in n_personas:
    pcs.append(cumple(11, n))

plt.plot(n_personas, pcs)
plt.show()

Podemos hacer más grande el tamaño de la tabla para reducir el número de colisiones. La probabilidad de colisión para n=18 y diferentes valores de m es

In [ ]:
emes = range(1, 1500, 100)
n = len(enteros)
probs = []
for m in emes:
    probs.append(cumple(m, n))

En lugar de hacer el cálculo de la probabilidad exacta con la función cumple, es posible aproximarla de la siguiente manera:

In [ ]:
def cumple_aprox(m, n):
    return 1.0 - exp((-n**2) / (2 * m))

aprox = []
for m in emes:
    aprox.append(cumple_aprox(m, n))

Graficamos las probabilidades de colisión con diferentes tamaños de tabla calculadas con las 2 estrategias

In [ ]:
plt.plot(emes, probs, label='$P[h(x) = h(y)]$')
plt.plot(emes, aprox, label='Aproximación')
plt.legend()
plt.show()

Asimismo, podemos calcular el número esperado de colisiones para valores dados de n y m de la siguiente manera

In [ ]:
def n_colisiones_esperado(m, n): 
    return n - m * (1 - ((m - 1) / m)**n)

Para una tabla con 11 cubetas y 18 números a almacenar esperaríamos el siguiente número de colisiones:

In [ ]:
n_colisiones_esperado(11, 18)

El número de colisiones que ocurrieron para el caso anterior fue:

In [ ]:
sum([len(c) - 1 for c in thmod_mas.tabla if len(c) > 1])

Si aumentamos el tamaño de la tabla a 281 cubetas esperaríamos

In [ ]:
n_cubetas = 281
n_colisiones_esperado(n_cubetas, 18)

Registremos los números en una tabla con este número de cubetas

In [ ]:
thmod_grande = THMod(n_cubetas)
for i,e in enumerate(enteros):
    thmod_grande.insertar(e)
print(thmod_grande)

Si tenemos más números para almacenar, el número esperado de colisiones sería:

In [ ]:
n_objetos = 200
n_colisiones_esperado(n_cubetas, n_objetos)

Veamos que pasa con nuestra tabla anterior si almacenamos 200 números

In [ ]:
conj = np.random.randint(1000, high=5890123, size=n_objetos)
thmod_mas_grande = THMod(n_cubetas)
for i,e in enumerate(conj):
    thmod_mas_grande.insertar(e)
print(thmod_mas_grande)

Examinemos la cantidad de números mapeados a cada cubeta por nuestra función hash

In [ ]:
cubetas = [c for c in range(n_cubetas)]
tam_cubetas = [len(c) for c in thmod_mas_grande.tabla]
plt.bar(cubetas, tam_cubetas)
plt.xlabel('Cubeta')
plt.ylabel(u'Número de colisiones')
plt.show()

El número de colisiones que ocurrieron

In [ ]:
sum([len(c) - 1 for c in thmod_mas_grande.tabla if len(c) > 1])

#### Direccionamiento abierto

En lugar de usar encadenamiento, podemos usar direccionamiento abierto resolver colisiones. En esta estrategia, si hay una colisión, se sondean otras cubetas o registros en la tabla hasta encontrar una libre. Si no hay ninguna disponible decimos que la tabla está llena y es necesario aumentar su tamaño para poder almacenar más objetos.

Para determinar el siguiente índice a examinar es necesario extender la función hash para que incorpore la información de índice. Esta función debe garantizar que se examinen todos los índices en caso de que ninguno esté disponible.

##### Sondeo lineal

In [ ]:
class THOA(TablaHash):
    def __init__(self, n_cubetas):
        super().__init__(n_cubetas)
        self.colisiones = np.zeros(n_cubetas)

    def sl(self, cubeta, i):
        return (cubeta + i) % self.n_cubetas

    def insertar(self, x):
        llena = True
        cubeta = self.h(x)  
        self.colisiones[cubeta] += 1
        for i in range(self.n_cubetas):
            ind = self.sl(cubeta, i)
            if not self.tabla[ind]:
                self.tabla[ind].append(x)
                llena = False
                break

        if llena:
            print('Tabla llena')

    def buscar(self, x):
        cubeta = self.h(x)
        for i in range(self.n_cubetas):
            ind = self.sl(cubeta, i)
            if self.tabla[ind][0] == x:
                return ind

        return -1

    def eliminar(self, x):
        cubeta = self.h(x)
        for i in range(self.n_cubetas):
            ind = self.sl(cubeta, i)
            if self.tabla[ind][0] == x:
                self.tabla[ind].remove(x)
                return ind

        return -1

class THModOA(THOA):
    def h(self, x):
        return x % self.n_cubetas 

Intentemos nuevamente con el conjunto anterior usando esta estrategia

In [ ]:
thmodoa = THModOA(n_cubetas)
for i,e in enumerate(conj):
    thmodoa.insertar(e)
print(thmodoa)

#### Familias por método de multiplicación

In [ ]:
class THMultOA(THOA):
    def __init__(self, n_cubetas, a):
        super().__init__(n_cubetas)
        self.a = a

    def h(self, x):
        return floor(((self.a * x) % 1) * self.n_cubetas) 

In [ ]:
thmultoa = THMultOA(281, (sqrt(5) - 1) / 2)
for i,e in enumerate(conj):
    thmultoa.insertar(e)
print(thmultoa)

La cantidad de números mapeados a cada cubeta por esta función hash es

In [ ]:
plt.bar(np.arange(thmultoa.colisiones.size), thmultoa.colisiones)
plt.xlabel('Cubeta')
plt.ylabel(u'Número de colisiones')
plt.show()

El total de colisiones que ocurrieron en la tabla fue

In [ ]:
np.sum(thmultoa.colisiones[thmultoa.colisiones > 1] - 1)

#### Familia de funciones hash universal

In [ ]:
class THUnivOA(THOA):
    def __init__(self, n_cubetas, a, b, primo):
        super().__init__(n_cubetas)
        self.a = a
        self.b = b
        self.primo = primo

    def h(self, x):
        return ((self.a * x + self.b) % self.primo) % self.n_cubetas



Instanciamos nuestra tabla con función hash universal y almacenamos el conjunto de 200 números


In [ ]:
a = random.randint(1, 104729 - 1)
b = random.randint(0, 104729 - 1)
thunivoa = THUnivOA(281, a, b, 104729)
for i,e in enumerate(conj):
    thunivoa.insertar(e)
print(thunivoa)

La cantidad de números mapeados a cada cubeta por la función hash universal es

In [ ]:
plt.bar(np.arange(thunivoa.colisiones.size), thunivoa.colisiones)
plt.xlabel('Cubeta')
plt.ylabel(u'Número de colisiones')
plt.show()

El número de colisiones totales que ocurrieron es

In [ ]:
np.sum(thunivoa.colisiones[thunivoa.colisiones > 1] - 1)

#### Funciones hash para cadenas

Modificamos nuestra clase anterior para que cuando haya una colisión verifique si es la misma subcadena.

In [ ]:
class THUnivCad(THUnivOA):
    def __init__(self, n_cubetas, a, b, primo, g):
        super().__init__(n_cubetas, a, b, primo)
        self.g = g

    def s2i(self, s):
        return sum([ord(c) * self.g**ord(c) for c in s])

    def insertar(self, s):
        x = int(self.s2i(s))
        cubeta = self.h(x)
        self.colisiones[cubeta] += 1

        llena = True
        for i in range(self.n_cubetas):
            ind = self.sl(cubeta, i)
            if not self.tabla[ind]:
                self.tabla[ind].append(s)
                llena = False
                break

        if llena:
            print('Tabla llena')

    def buscar(self, s):
        x = self.s2i(s)
        cubeta = self.h(x)
        for i in range(self.n_cubetas):
            ind = self.sl(cubeta, i)
            if self.tabla[ind][0] == s:
                return ind

        return -1

    def eliminar(self, s):
        x = self.s2i(s)
        cubeta = self.h(x)
        for i in range(self.n_cubetas):
            ind = self.sl(cubeta, i)
            if self.tabla[ind][0] == s:
                self.tabla[ind].remove(s)
                return ind

        return -1

In [ ]:
nombres = ['blanca', 'juan', 'pedro', 'julia', 'serena', 'enrique', 'jose', 
           'alfonso', 'rodolfo', 'ignacio', 'mariana']
thucad = THUnivCad(281, a, b, 104729, 1.1)
for e in nombres:
    thucad.insertar(e)
print(thucad)

Revisemos la cantidad de nombres mapeados a cada cubeta

In [ ]:
plt.bar(np.arange(thucad.colisiones.size), thucad.colisiones)
plt.xlabel('Cubeta')
plt.ylabel(u'Número de colisiones')
plt.show()